In [1]:
# --- mount the dataset from drive
import os
from google.colab import drive

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

if not os.path.isdir('/content/doppler_traces'):
    !unzip "/content/drive/MyDrive/DATASET_SHARP/doppler_traces.zip"

Mounted at /content/drive
Archive:  /content/drive/MyDrive/DATASET_SHARP/doppler_traces.zip
   creating: doppler_traces/
  inflating: doppler_traces/.placeholder  
   creating: doppler_traces/S1a/
  inflating: doppler_traces/S1a/S1a_C_stream_3.txt  
  inflating: doppler_traces/S1a/S1a_H_stream_2.txt  
  inflating: doppler_traces/S1a/S1a_W_stream_1.txt  
  inflating: doppler_traces/S1a/S1a_E_stream_3.txt  
  inflating: doppler_traces/S1a/S1a_R_stream_2.txt  
  inflating: doppler_traces/S1a/S1a_J2_stream_0.txt  
  inflating: doppler_traces/S1a/S1a_J1_stream_3.txt  
  inflating: doppler_traces/S1a/S1a_S_stream_3.txt  
  inflating: doppler_traces/S1a/S1a_J1_stream_0.txt  
  inflating: doppler_traces/S1a/S1a_S_stream_2.txt  
  inflating: doppler_traces/S1a/S1a_C_stream_1.txt  
  inflating: doppler_traces/S1a/S1a_J2_stream_1.txt  
  inflating: doppler_traces/S1a/S1a_J2_stream_3.txt  
  inflating: doppler_traces/S1a/S1a_J1_stream_2.txt  
  inflating: doppler_traces/S1a/S1a_E_stream_1.txt  
  

## UTILITY

In [2]:
import glob
import os
import numpy as np
import pickle
import math as mt
import shutil

def convert_to_number(lab, csi_label_dict):
    lab_num = np.argwhere(np.asarray(csi_label_dict) == lab)[0][0]
    return lab_num


def create_windows(csi_list, labels_list, sample_length, stride_length):
    csi_matrix_stride = []
    labels_stride = []
    for i in range(len(labels_list)):
        csi_i = csi_list[i]
        label_i = labels_list[i]
        len_csi = csi_i.shape[1]
        for ii in range(0, len_csi - sample_length, stride_length):
            csi_matrix_stride.append(csi_i[:, ii:ii+sample_length])
            labels_stride.append(label_i)
    return csi_matrix_stride, labels_stride

def create_windows_antennas(csi_list, labels_list, sample_length, stride_length, remove_mean=False):
    # -------------------------------------------------------------------------------------
    #  Split each activity into windows (keeping all antennas together)
    csi_matrix_stride = []
    labels_stride = []
    for i in range(len(labels_list)):        # iterate over the N ACTIVITIES
        csi_i = csi_list[i]                  # Doppler spectrum (all antennas) of activity i
        label_i = labels_list[i]             # its label (a single value)
        len_csi = csi_i.shape[2]             # time length = number of available Doppler columns
        # Sliding window: starts at 0, advances by stride_length, stops when
        # there is no more room for a full window of sample_length columns.
        for ii in range(0, len_csi - sample_length, stride_length):
            csi_wind = csi_i[:, :, ii:ii + sample_length, ...]   # crop [ii, ii+sample_length)
            if remove_mean:
                # Optional normalization: mean over time (axis 2) and subtraction
                # -> removes the window's constant offset (here it is False by default).
                csi_mean = np.mean(csi_wind, axis=2, keepdims=True)
                csi_wind = csi_wind - csi_mean
            csi_matrix_stride.append(csi_wind)   # +1 sample (one window)
            labels_stride.append(label_i)        # +1 label = the one of activity i
    return csi_matrix_stride, labels_stride

## GENERATE DATASET 

In [ ]:
# =====================================================================================
#  DATASET GENERATION (single train/test method)
#  Refactoring of CSI_doppler_create_dataset_train.py and _test.py into ONE function.
#  The `complete` parameter selects the behavior:
#    * complete=False -> like CSI_doppler_create_dataset_train.py:
#                        temporal 60/20/20 split per activity -> train/val/test folders
#    * complete=True  -> like CSI_doppler_create_dataset_test.py:
#                        NO split, the whole sequence -> single "complete" folder
# =====================================================================================

def generate_dataset(data_dir, list_subdir, num_packets, sliding, window_length,
                   stride_length, labels_activities, n_tot, complete=False):
    """
    data_dir         : root folder of the data (Doppler spectra)
    list_subdir      : experiment subfolders, comma-separated (e.g. "S1a,S1b,S1c")
    num_packets      : n. of Wi-Fi packets used for one STFT (e.g. 31)
    sliding          : step (in packets) between two consecutive STFTs
    window_length    : n. of Doppler columns per window (network input, e.g. 340)
    stride_length    : step between two consecutive windows
    labels_activities: activities considered, e.g. "E,J,L,R,W" (the index becomes the label)
    n_tot            : n. of spatial streams * n. of antennas (channels per activity, e.g. 4)
    complete         : False -> train/val/test split ; True -> single "complete" set
    """
    # --- Label dictionary: from "E,J,L,R,W" to a list; the index = numeric label -------
    csi_label_dict = []
    for lab_act in labels_activities.split(','):
        csi_label_dict.append(lab_act)
    # String used only to build the names of output folders/files.
    activities = np.asarray(labels_activities)

    # ==================================================================================
    #  MAIN LOOP: one subfolder (one experiment) at a time
    # ==================================================================================
    for subdir in list_subdir.split(','):
        exp_dir = data_dir + subdir + '/'  # full path of the experiment folder

        # Output folder names
        path_train = exp_dir + 'train_antennas_' + str(activities)
        path_val = exp_dir + 'val_antennas_' + str(activities)
        path_test = exp_dir + 'test_antennas_' + str(activities)
        path_complete = exp_dir + 'complete_antennas_' + str(activities)

        # --- Folder preparation (different logic in the two modes) --------------------
        if not complete:
            # TRAIN MODE: prepare train/val/test (empty them if they exist, else create)
            for pat in [path_train, path_val, path_test]:
                if os.path.exists(pat):
                    for f in glob.glob(pat + '/*'):
                        os.remove(f)
                else:
                    os.mkdir(pat)
            # remove any "complete" from previous runs
            if os.path.exists(path_complete):
                shutil.rmtree(path_complete)
        else:
            # TEST MODE: fully delete train/val/test and prepare "complete"
            # NB: mirrors the original script -> do not generate 'complete' on the same
            #     subdir used for training, or you would erase its split.
            for pat in [path_train, path_val, path_test]:
                if os.path.exists(pat):
                    shutil.rmtree(pat)
            if os.path.exists(path_complete):
                for f in glob.glob(path_complete + '/*'):
                    os.remove(f)
            else:
                os.mkdir(path_complete)

        # --- Input file collection (common to both modes) ----------------------------
        # Files starting with 'S', without extension (.txt), SORTED: this way the n_tot
        # antennas of the same activity end up consecutive.
        names = []
        all_files = os.listdir(exp_dir)
        for i in range(len(all_files)):
            if all_files[i].startswith('S'):
                names.append(all_files[i][:-4])
        names.sort()

        # --- Grouping by activity (common to both modes) -----------------------------
        # Each activity = n_tot consecutive files (one per antenna). Accumulate into
        # 'csi_matrix' and, when complete, pack into 'csi_matrices' with the label.
        csi_matrices = []   # list of arrays (n_tot, doppler_freq, time) -> one per activity
        labels = []         # numeric label of each activity
        lengths = []        # temporal duration (n. of Doppler columns) of each activity
        label = 'null'
        prev_label = label
        csi_matrix = []     # buffer: the antennas of the current activity
        processed = False
        for i_name, name in enumerate(names):
            # At every multiple of n_tot (except the first) an antenna group is complete.
            if i_name % n_tot == 0 and i_name != 0 and processed:
                ll = csi_matrix[0].shape[1]
                for i_ant in range(1, n_tot):
                    if ll != csi_matrix[i_ant].shape[1]:
                        break
                lengths.append(ll)
                csi_matrices.append(np.asarray(csi_matrix))  # stack -> (n_tot, freq, time)
                labels.append(label)
                csi_matrix = []

            label = name[4]  # the label = 5th character of the file name
            if label not in csi_label_dict:
                processed = False
                continue
            processed = True
            print(name)

            label = convert_to_number(label, csi_label_dict)
            if i_name % n_tot == 0:
                prev_label = label
            elif label != prev_label:
                print('error in ' + str(name))
                break

            # Load the Doppler spectrum of the single antenna and remove the per-column mean.
            name_file = exp_dir + name + '.txt'
            with open(name_file, "rb") as fp:
                stft_sum_1 = pickle.load(fp)
            stft_sum_1_mean = stft_sum_1 - np.mean(stft_sum_1, axis=0, keepdims=True)
            csi_matrix.append(stft_sum_1_mean.T)  # -> (doppler_freq, time)

        # --- Closing of the LAST group (common to both modes) -------------------------
        error = False
        if processed:
            if len(csi_matrix) < n_tot:
                print('error in ' + str(name))
            ll = csi_matrix[0].shape[1]
            for i_ant in range(1, n_tot):
                if ll != csi_matrix[i_ant].shape[1]:
                    print('error in ' + str(name))
                    error = True
            if not error:
                lengths.append(ll)
                csi_matrices.append(np.asarray(csi_matrix))
                labels.append(label)

        if error:
            continue

        # ==============================================================================
        #  DISPATCH: split (train) or whole sequence (complete)
        # ==============================================================================
        if not complete:
            # ----- TRAIN MODE: temporal 60/20/20 split per activity -------------------
            lengths = np.asarray(lengths)
            csi_train, csi_val, csi_test = [], [], []
            length_train, length_val, length_test = [], [], []
            for i in range(len(labels)):
                ll = lengths[i]
                # TRAIN: first 60%
                train_len = int(np.floor(ll * 0.6))
                length_train.append(train_len)
                csi_train.append(csi_matrices[i][:, :, :train_len])
                # VAL: next 20%, with a ceil(num_packets/sliding) gap to avoid overlap
                start_val = train_len + mt.ceil(num_packets / sliding)
                val_len = int(np.floor(ll * 0.2))
                length_val.append(val_len)
                csi_val.append(csi_matrices[i][:, :, start_val:start_val + val_len])
                # TEST: remainder, after another gap
                start_test = start_val + val_len + mt.ceil(num_packets / sliding)
                length_test.append(ll - val_len - train_len - 2 * mt.ceil(num_packets / sliding))
                csi_test.append(csi_matrices[i][:, :, start_test:])

            list_sets_name = ['train', 'val', 'test']
            list_sets = [csi_train, csi_val, csi_test]
            list_sets_lengths = [length_train, length_val, length_test]

            # Windowing + saving for each of the 3 sets
            for set_idx in range(3):
                csi_matrices_set, labels_set = create_windows_antennas(
                    list_sets[set_idx], labels, window_length, stride_length, remove_mean=False)
                num_windows = np.floor(
                    (np.asarray(list_sets_lengths[set_idx]) - window_length) / stride_length + 1)
                if not len(csi_matrices_set) == np.sum(num_windows):
                    print('ERROR - shapes mismatch')
                _save_windows(exp_dir, list_sets_name[set_idx], str(activities),
                                csi_matrices_set, labels_set, num_windows)
        else:
            # ----- TEST MODE: no split, the whole sequence -> "complete" --------------
            csi_complete = [csi_matrices[i] for i in range(len(labels))]
            csi_matrices_wind, labels_wind = create_windows_antennas(
                csi_complete, labels, window_length, stride_length, remove_mean=False)
            num_windows = np.floor((np.asarray(lengths) - window_length) / stride_length + 1)
            if not len(csi_matrices_wind) == np.sum(num_windows):
                print('ERROR - shapes mismatch')
            _save_windows(exp_dir, 'complete', str(activities),
                            csi_matrices_wind, labels_wind, num_windows)


def _save_windows(exp_dir, set_name, activities_str, csi_windows, labels_windows, num_windows):
    """Save the windows of a set + the summary files (labels_/files_/num_windows_)."""
    suffix = '.txt'
    names_set = []
    for ii in range(len(csi_windows)):
        name_file = exp_dir + set_name + '_antennas_' + activities_str + '/' + str(ii) + suffix
        names_set.append(name_file)
        with open(name_file, "wb") as fp:            # one window per file (pickle)
            pickle.dump(csi_windows[ii], fp)
    # labels_*: label of each window
    with open(exp_dir + '/labels_' + set_name + '_antennas_' + activities_str + suffix, "wb") as fp:
        pickle.dump(labels_windows, fp)
    # files_*: paths of the saved windows (used by the loader)
    with open(exp_dir + '/files_' + set_name + '_antennas_' + activities_str + suffix, "wb") as fp:
        pickle.dump(names_set, fp)
    # num_windows_*: n. of windows per activity (to reconstruct the antenna groups)
    with open(exp_dir + '/num_windows_' + set_name + '_antennas_' + activities_str + suffix, "wb") as fp:
        pickle.dump(num_windows, fp)


# =====================================================================================
#  PARAMETERS + CALL (for now in NON complete mode -> train/val/test split)
# =====================================================================================
data_dir = '/content/doppler_traces/'   # root folder of the data (dataset unpacked on Colab)
list_subdir = 'S1a,S1b,S1c'             # experiment subfolders (comma-separated)
num_packets = 31                        # n. of Wi-Fi packets per STFT
sliding = 1                             # step (in packets) between two STFTs
window_length = 340                     # n. of Doppler columns per window (network input)
stride_length = 30                      # step between two consecutive windows
labels_activities = 'E,J,L,R,W'         # activities (the index = label: E->0, J->1, ...)
n_tot = 4                               # n. of streams * n. of antennas

# Requested call: NON complete mode (generates train/val/test)
generate_dataset(data_dir, list_subdir, num_packets, sliding, window_length,
               stride_length, labels_activities, n_tot, complete=False)

## NEURAL NETWORK — CSI_network (PyTorch)
PyTorch porting of `CSI_network.py` + `network_utility.py` + `dataset_utility.py` (single-antenna loader). The points where we switch from TensorFlow/Keras to PyTorch are marked with `>>> CHANGE`.

In [ ]:
# =====================================================================================
#  NEURAL NETWORK — porting of network_utility.py (Keras) to PyTorch
#  "inception-residual" single-antenna architecture (csi_network_inc_res).
#  >>> CHANGE TF->PyTorch: Keras uses channels-LAST (N,H,W,C); PyTorch channels-FIRST
#      (N,C,H,W). The network input goes from (340,100,1) [Keras] to (1,340,100) [PyTorch].
# =====================================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F


class ConvBnRelu(nn.Module):
    """
    Porting of conv2d_bn: in the original it was Conv2D + (optional BatchNorm) + ReLU.
    NB: in the SHARP model bn=False always -> here it is just Conv2d + ReLU.
    >>> CHANGE: tf.keras.layers.Conv2D -> nn.Conv2d (channels first).
    """
    def __init__(self, in_ch, out_ch, kernel_size, stride=1, padding='same', activation=True, batchNormalization = False):
        super().__init__()
        # Keras 'same': in PyTorch supported ONLY with stride=1. For the convs with
        # stride=2 we pass an integer padding computed by hand (see ReductionABlockSmall).
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size, stride=stride, padding=padding)
        self.bn = nn.BatchNorm2d(out_ch)
        self.batchNormalization = batchNormalization
        self.activation = activation

    def forward(self, x):
        x = self.conv(x)
        if self.batchNormalization:
            x = self.bn(x)
        if self.activation:
            x = F.relu(x)   # >>> CHANGE: Activation('relu') -> F.relu
        return x


class ReductionABlockSmall(nn.Module):
    """
    Porting of reduction_a_block_small: 3 parallel branches concatenated over channels.
    Input (N,1,340,100) -> output (N,15,170,50).
    """
    def __init__(self, in_ch):
        super().__init__()
        # Branch 1: MaxPool 2x2 stride 2 (padding 'valid'=0) -> halves H,W, 1 channel.
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        # Branch 2: Conv 5 filters 2x2 stride 2 padding 'valid'(=0) -> (N,5,170,50).
        self.b2 = ConvBnRelu(in_ch, 5, kernel_size=2, stride=2, padding=0)
        # Branch 3: 1x1 same -> 2x2 same -> 4x4 stride2.
        self.b3_1 = ConvBnRelu(in_ch, 3, kernel_size=1, stride=1, padding='same')
        self.b3_2 = ConvBnRelu(3, 6, kernel_size=2, stride=1, padding='same')
        # >>> CHANGE: TF's 4x4 stride2 'same' conv -> padding=1. TF's 'same' here
        #     requires total padding = 2 (symmetric), so padding=1 replicates
        #     EXACTLY the TensorFlow 170x50 output ('same' not usable with stride 2).
        self.b3_3 = ConvBnRelu(6, 9, kernel_size=4, stride=2, padding=1)

    def forward(self, x):
        x1 = self.pool(x)          # (N,1,170,50)
        x2 = self.b2(x)            # (N,5,170,50)
        x3 = self.b3_1(x)
        x3 = self.b3_2(x3)
        x3 = self.b3_3(x3)         # (N,9,170,50)
        # >>> CHANGE: Concatenate (trailing channels axis in Keras) -> torch.cat(dim=1).
        return torch.cat([x1, x2, x3], dim=1)   # (N,15,170,50)


class CSINetworkIncRes(nn.Module):
    """PyTorch equivalent of csi_network_inc_res(input_sh, output_sh)."""
    def __init__(self, input_channels, num_classes, input_hw=(340, 100)):
        super().__init__()
        self.reduction = ReductionABlockSmall(input_channels)
        # conv4: 3 filters 1x1 'same' + ReLU (reduces 15 -> 3 feature maps).
        self.conv4 = ConvBnRelu(15, 3, kernel_size=1, stride=1, padding='same')
        # Flatten dimension computed from the shapes (170x50 after the reduction).
        h, w = input_hw
        h2, w2 = h // 2, w // 2          # 170, 50
        self.flat_dim = 3 * h2 * w2      # 3*170*50 = 25500
        self.dropout = nn.Dropout(0.2)
        # >>> CHANGE: Dense(output_sh, activation=None) -> nn.Linear (LOGITS, no softmax).
        self.fc = nn.Linear(self.flat_dim, num_classes)
        self._init_weights()

    def _init_weights(self):
        # >>> CHANGE: Keras initializes Conv/Dense with Glorot uniform by default,
        #     PyTorch with Kaiming. We apply Xavier/Glorot to stay faithful.
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.reduction(x)
        x = self.conv4(x)
        x = torch.flatten(x, start_dim=1)   # Flatten keeping the batch dim
        x = self.dropout(x)
        x = self.fc(x)                      # logits
        return x

## DATASET LOADER (PyTorch)

In [ ]:
# =====================================================================================
#  DATASET LOADER — porting of dataset_utility.py (tf.data) to PyTorch
#  >>> CHANGE TF->PyTorch: tf.data.Dataset + tf.numpy_function -> Dataset + DataLoader.
#      No need for .cache()/.repeat()/.prefetch(): the DataLoader handles shuffle,
#      batch and prefetch, and it is iterated once per epoch (no .repeat()).
# =====================================================================================
from torch.utils.data import Dataset, DataLoader


def expand_antennas(file_names, labels, num_antennas):
    # UNCHANGED (pure numpy logic): each window -> num_antennas single-antenna samples.
    file_names_expanded = [item for item in file_names for _ in range(num_antennas)]
    labels_expanded = [item for item in labels for _ in range(num_antennas)]
    stream_ant = np.tile(np.arange(num_antennas), len(labels))
    return file_names_expanded, labels_expanded, stream_ant


class CSIDatasetSingle(Dataset):
    """Porting of load_data_single + create_dataset_single (single-antenna input)."""
    def __init__(self, csi_files, labels, stream_ant):
        self.files = csi_files
        self.labels = labels
        self.stream_ant = list(stream_ant)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        # Load the window (all antennas) and select the requested antenna.
        with open(self.files[idx], "rb") as fp:
            matrix_csi = pickle.load(fp)          # (n_tot, doppler=100, time=340)
        stream = self.stream_ant[idx]
        m = matrix_csi[stream, ...].T             # 1 antenna, transposed -> (340, 100)
        # >>> CHANGE: in Keras the channel went LAST -> (340,100,1).
        #     In PyTorch the channel goes FIRST -> (1,340,100) (NCHW format).
        m = np.ascontiguousarray(m[np.newaxis, ...])   # (1, 340, 100)
        x = torch.from_numpy(m).float()           # >>> CHANGE: tf.cast(float32) -> .float()
        # integer label (long): required by nn.CrossEntropyLoss (not one-hot).
        y = torch.tensor(int(self.labels[idx]), dtype=torch.long)
        return x, y


def make_dataloader(csi_files, labels, stream_ant, batch_size, shuffle):
    # >>> CHANGE: create_dataset_single(...) -> DataLoader (shuffle=True only in training).
    ds = CSIDatasetSingle(csi_files, labels, stream_ant)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=2, drop_last=False)

## TRAINING (PyTorch)

In [ ]:
# =====================================================================================
#  TRAINING — porting of the CSI_network.py main to PyTorch
# =====================================================================================
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, accuracy_score

# --- Parameters (consistent with the dataset generation) ----------------------------
subdirs_training = 'S1a,S1b,S1c'   # sessions used for train/val/test
csi_act = labels_activities        # MUST match the generated folder names ('E,J,L,R,W')
feature_length = 100               # Doppler bins (feature height)
sample_length = 340                # time (width)
channels = 1                       # single antenna
batch_size = 32
num_antennas = n_tot               # 4
name_base = 'single_ant'

activities = np.asarray([a for a in csi_act.split(',')])
output_shape = activities.shape[0]                 # n. of classes = output neurons
labels_considered = np.arange(output_shape)        # [0..n_classes-1]
suffix = '.txt'

# --- Loading the indices (labels/files) for each session ----------------------------
# UNCHANGED: only the window paths are loaded (via pickle), not the data.
labels_train, all_files_train = [], []
labels_val,   all_files_val   = [], []
labels_test,  all_files_test  = [], []
for sdir in subdirs_training.split(','):
    base = data_dir + sdir + '/'
    with open(base + 'labels_train_antennas_' + str(csi_act) + suffix, "rb") as fp:
        labels_train.extend(pickle.load(fp))
    with open(base + 'files_train_antennas_' + str(csi_act) + suffix, "rb") as fp:
        all_files_train.extend(pickle.load(fp))
    with open(base + 'labels_val_antennas_' + str(csi_act) + suffix, "rb") as fp:
        labels_val.extend(pickle.load(fp))
    with open(base + 'files_val_antennas_' + str(csi_act) + suffix, "rb") as fp:
        all_files_val.extend(pickle.load(fp))
    with open(base + 'labels_test_antennas_' + str(csi_act) + suffix, "rb") as fp:
        labels_test.extend(pickle.load(fp))
    with open(base + 'files_test_antennas_' + str(csi_act) + suffix, "rb") as fp:
        all_files_test.extend(pickle.load(fp))

# --- Class filter + antenna expansion (train/val/test) ------------------------------
def filter_and_expand(all_files, labels):
    files_sel = [all_files[i] for i in range(len(labels)) if labels[i] in labels_considered]
    labels_sel = [labels[i] for i in range(len(labels)) if labels[i] in labels_considered]
    files_exp, labels_exp, stream_ant = expand_antennas(files_sel, labels_sel, num_antennas)
    return files_sel, labels_sel, files_exp, labels_exp, stream_ant

file_train_sel, labels_train_sel, file_train_exp, labels_train_exp, stream_ant_train = \
    filter_and_expand(all_files_train, labels_train)
file_val_sel, labels_val_sel, file_val_exp, labels_val_exp, stream_ant_val = \
    filter_and_expand(all_files_val, labels_val)
file_test_sel, labels_test_sel, file_test_exp, labels_test_exp, stream_ant_test = \
    filter_and_expand(all_files_test, labels_test)

# --- DataLoader: shuffle only in training -------------------------------------------
train_loader = make_dataloader(file_train_exp, labels_train_exp, stream_ant_train, batch_size, shuffle=True)
val_loader   = make_dataloader(file_val_exp,   labels_val_exp,   stream_ant_val,   batch_size, shuffle=False)
test_loader  = make_dataloader(file_test_exp,  labels_test_exp,  stream_ant_test,  batch_size, shuffle=False)

# --- Model build, loss, optimizer ---------------------------------------------------
# >>> CHANGE: tf.config.list_physical_devices('GPU') -> PyTorch device selection.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

csi_model = CSINetworkIncRes(input_channels=channels, num_classes=output_shape,
                             input_hw=(sample_length, feature_length)).to(device)
print(csi_model)  # >>> CHANGE: model.summary() -> print(model)

# >>> CHANGE: Adam(learning_rate=0.0001) -> torch.optim.Adam(..., lr=1e-4)
optimizer = torch.optim.Adam(csi_model.parameters(), lr=1e-4)
# >>> CHANGE: SparseCategoricalCrossentropy(from_logits=True) -> nn.CrossEntropyLoss
#     (applies log-softmax internally: the model outputs LOGITS, no softmax).
criterion = nn.CrossEntropyLoss()

# Checkpoint file (.torch extension, as in the course material).
name_model = name_base + '_' + str(csi_act) + '_network.torch'

# =====================================================================================
#  TRAINING LOOP (max 25 epochs) with a FULL, RESUMABLE CHECKPOINT
#  Course guidance: Colab stops long sessions, so we save a checkpoint that lets us
#  RESUME training later. To resume seamlessly we must store BOTH the network weights
#  AND the optimizer state (Adam moments) — the network alone would suffice only for
#  inference/evaluation.
#  >>> CHANGE: model.fit(..., callbacks=[EarlyStopping, ModelCheckpoint]) -> manual loop
#      - EarlyStopping(monitor='val_loss', patience=3) -> manual count on val_loss
#      - ModelCheckpoint(save_best_only, monitor='val_accuracy') -> keep best weights
# =====================================================================================
epochs = 25
patience = 3

# --- Resume state (defaults for a fresh run) ----------------------------------------
start_epoch = 0
best_val_acc = -1.0            # best value (for the checkpoint, monitor accuracy)
best_val_loss = float('inf')  # best value (for early stopping, monitor val_loss)
epochs_no_improve = 0
best_model_state = None        # weights of the best epoch (reloaded for evaluation)

# --- If a checkpoint already exists, RESUME from it ---------------------------------
if os.path.exists(name_model):
    ckpt = torch.load(name_model, map_location=device)
    csi_model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])   # restore the Adam moments
    start_epoch = ckpt['epoch'] + 1
    best_val_acc = ckpt['best_val_acc']
    best_val_loss = ckpt['best_val_loss']
    epochs_no_improve = ckpt['epochs_no_improve']
    best_model_state = ckpt['best_model_state']
    print(f'Resuming from epoch {start_epoch} (best_val_acc={best_val_acc:.4f})')

for epoch in range(start_epoch, epochs):
    # ---- Training phase ----
    csi_model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()          # >>> CHANGE: zero the gradients (implicit in Keras)
        logits = csi_model(xb)
        loss = criterion(logits, yb)
        loss.backward()                # >>> CHANGE: explicit backpropagation
        optimizer.step()               # >>> CHANGE: explicit weight update

    # ---- Validation phase ----
    csi_model.eval()
    val_loss_sum, correct, total = 0.0, 0, 0
    with torch.no_grad():              # >>> CHANGE: no gradients during validation
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = csi_model(xb)
            val_loss_sum += criterion(logits, yb).item() * xb.size(0)
            correct += (logits.argmax(1) == yb).sum().item()
            total += yb.size(0)
    val_loss = val_loss_sum / total
    val_acc = correct / total
    print(f'Epoch {epoch + 1}/{epochs} - val_loss={val_loss:.4f} - val_acc={val_acc:.4f}')

    # Track the best weights (monitor val_acc) — kept inside the checkpoint.
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_state = {k: v.cpu().clone() for k, v in csi_model.state_dict().items()}
    # Early stopping bookkeeping on val_loss (patience=3).
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    # --- FULL CHECKPOINT saved every epoch (network + optimizer + resume state) ------
    torch.save({
        'epoch': epoch,
        'model_state_dict': csi_model.state_dict(),        # current weights (to resume)
        'optimizer_state_dict': optimizer.state_dict(),    # Adam moments (to resume)
        'best_val_acc': best_val_acc,
        'best_val_loss': best_val_loss,
        'epochs_no_improve': epochs_no_improve,
        'best_model_state': best_model_state,              # best weights so far (for eval)
    }, name_model)

    if epochs_no_improve >= patience:
        print('Early stopping')
        break

# Reload the BEST weights for evaluation (like load_model after fit).
csi_model.load_state_dict(best_model_state)

## EVALUATION + ANTENNA MERGE (PyTorch)

In [ ]:
# =====================================================================================
#  EVALUATION + ANTENNA MERGE (decision fusion) — porting from CSI_network.py
# =====================================================================================

# >>> CHANGE TF->PyTorch: model.predict(dataset) -> manual loop with eval()+no_grad()
#     that returns the concatenated logits.
def predict_logits(model, loader):
    model.eval()
    out = []
    with torch.no_grad():
        for xb, _ in loader:
            out.append(model(xb.to(device)).cpu().numpy())
    return np.concatenate(out, axis=0)

# --- Per-antenna predictions on the TEST set ----------------------------------------
test_labels_true = np.array(labels_test_exp)
# [:N] trims any extra samples (in PyTorch without .repeat() there are none,
#      but we keep the trim for consistency with the original).
test_prediction_list = predict_logits(csi_model, test_loader)[:test_labels_true.shape[0]]
test_labels_pred = np.argmax(test_prediction_list, axis=1)   # predicted class per-antenna

# --- Single-antenna level metrics (sklearn: UNCHANGED from the original) -------------
conf_matrix = confusion_matrix(test_labels_true, test_labels_pred)
precision, recall, fscore, _ = precision_recall_fscore_support(
    test_labels_true, test_labels_pred, labels=labels_considered)
accuracy = accuracy_score(test_labels_true, test_labels_pred)
print('Per-antenna accuracy:', accuracy)
print('Confusion matrix (per-antenna):\n', conf_matrix)

# =====================================================================================
#  ANTENNA MERGE (decision fusion) on the TEST set — UNCHANGED (pure numpy logic)
#  For each acquisition combine the num_antennas per-antenna predictions:
#    - majority vote on the labels;
#    - in case of a tie or >2 classes -> argmax of the SUM of the logits.
# =====================================================================================
labels_true_merge = np.array(labels_test_sel)          # 1 label per acquisition
pred_max_merge = np.zeros(len(labels_test_sel), dtype=int)
for i_lab in range(len(labels_test_sel)):
    # Block of num_antennas rows = the predictions of the 4 antennas of the same acquisition.
    pred_antennas = test_prediction_list[i_lab * num_antennas:(i_lab + 1) * num_antennas, :]
    lab_merge_max = np.argmax(np.sum(pred_antennas, axis=0))   # argmax of the sum of the logits

    pred_max_antennas = test_labels_pred[i_lab * num_antennas:(i_lab + 1) * num_antennas]
    lab_unique, count = np.unique(pred_max_antennas, return_counts=True)
    if lab_unique.shape[0] > 1:                 # the antennas do NOT all agree
        count_argsort = np.flip(np.argsort(count))
        count_sort = count[count_argsort]
        lab_unique_sort = lab_unique[count_argsort]
        if count_sort[0] == count_sort[1] or lab_unique.shape[0] > 2:  # tie or >2 classes
            lab_max_merge = lab_merge_max        # fallback: sum of the logits
        else:
            lab_max_merge = lab_unique_sort[0]   # the most voted class wins
    else:
        lab_max_merge = lab_unique[0]            # all the antennas agree
    pred_max_merge[i_lab] = lab_max_merge

# --- Acquisition-level metrics (after the antenna fusion) ---------------------------
conf_matrix_max_merge = confusion_matrix(labels_true_merge, pred_max_merge, labels=labels_considered)
precision_max_merge, recall_max_merge, fscore_max_merge, _ = precision_recall_fscore_support(
    labels_true_merge, pred_max_merge, labels=labels_considered)
accuracy_max_merge = accuracy_score(labels_true_merge, pred_max_merge)
print('Accuracy after antenna merge:', accuracy_max_merge)
print('Confusion matrix (merge):\n', conf_matrix_max_merge)